# 02 — Component and Subcomponent Extraction

## Purpose

Parse World Bank PAD text into component- and subcomponent-level analytical units.

This notebook:

1. Loads downloaded PAD text files.
2. Detects component and subcomponent headings.
3. Extracts the corresponding text blocks.
4. Preserves extraction failures for review.
5. Saves structured analytical units for embedding and retrieval.

In [ ]:
from pathlib import Path
import re

import pandas as pd

ROOT = Path("..").resolve()

RAW_PAD_DIR = ROOT / "data" / "raw" / "pad"
PROCESSED_DIR = ROOT / "data" / "processed"
META_DIR = ROOT / "data" / "metadata"

SAMPLE_PATH = META_DIR / "prototype_pad_sample.csv"
UNITS_PATH = PROCESSED_DIR / "pad_components_subcomponents.csv"
SUMMARY_PATH = PROCESSED_DIR / "pad_extraction_summary.csv"
FAILURES_PATH = PROCESSED_DIR / "pad_extraction_failures.csv"

MAX_BLOCK_CHARS = 50_000

In [ ]:
# --------------------------------------------------
# LOAD MANIFEST AND TEXT FILES
# --------------------------------------------------
sample_df = pd.read_csv(SAMPLE_PATH)
text_files = sorted(PAD_DIR.glob("*.txt"))

print("Documents in manifest:", len(sample_df))
print("Downloaded text files:", len(text_files))

In [ ]:
# --------------------------------------------------
# HEADING DETECTION PATTERNS
# --------------------------------------------------

COMPONENT_PATTERN = re.compile(
    r"""
    ^\s*
    (?:\d+\.\s*)?
    Component
    \s+
    (?P<number>\d+)
    \s*[:\.\-]\s*
    (?P<title>.+?)
    \s*$
    """,
    re.IGNORECASE | re.VERBOSE,
)

SUBCOMPONENT_PATTERN = re.compile(
    r"""
    ^\s*
    (?:\d+\.\s*)?
    Sub[- ]?component
    \s+
    (?P<number>\d+\.\d+)
    \s*[:\.\-]\s*
    (?P<title>.+?)
    \s*$
    """,
    re.IGNORECASE | re.VERBOSE,
)

In [ ]:
# --------------------------------------------------
# EXTRACTION FUNCTIONS
# --------------------------------------------------

def detect_headings(text):
    """Detect numbered component and subcomponent headings in PAD text."""
    headings = []
    for i, line in enumerate(text.splitlines()):
        cm = COMPONENT_PATTERN.match(line)
        sm = SUBCOMPONENT_PATTERN.match(line)
        if cm:
            headings.append({
                "line_number": i + 1, "line_index": i,
                "type": "component",
                "number": cm.group("number"),
                "title": cm.group("title").strip(),
            })
        elif sm:
            headings.append({
                "line_number": i + 1, "line_index": i,
                "type": "subcomponent",
                "number": sm.group("number"),
                "title": sm.group("title").strip(),
            })
    return pd.DataFrame(headings)


def extract_heading_blocks(text, headings):
    """Extract the text block following each heading."""
    lines  = text.splitlines()
    blocks = []
    for i, heading in headings.iterrows():
        start = heading["line_index"] + 1
        end   = headings.iloc[i + 1]["line_index"] if i < len(headings) - 1 else len(lines)
        block_text = "\n".join(lines[start:end]).strip()
        blocks.append({
            "type":            heading["type"],
            "number":          heading["number"],
            "title":           heading["title"],
            "line_number":     heading["line_number"],
            "text":            block_text,
            "character_count": len(block_text),
        })
    return pd.DataFrame(blocks)


def extract_canonical_blocks(text):
    """Return the earliest valid occurrence of each numbered component or subcomponent."""
    headings_df = detect_headings(text)
    if headings_df.empty:
        return pd.DataFrame()
    blocks_df       = extract_heading_blocks(text, headings_df)
    valid_blocks_df = blocks_df[blocks_df["character_count"] <= MAX_BLOCK_CHARACTERS].copy()
    if valid_blocks_df.empty:
        valid_blocks_df = blocks_df.copy()
    return (
        valid_blocks_df
        .sort_values("line_number")
        .drop_duplicates(subset=["type", "number"], keep="first")
        .sort_values(["type", "number"])
        .reset_index(drop=True)
    )

In [ ]:
# --------------------------------------------------
# EXTRACT ACROSS FULL CORPUS
# --------------------------------------------------

all_extracted_blocks = []
document_summary     = []

for text_path in text_files:
    text       = text_path.read_text(encoding="utf-8", errors="replace")
    blocks_df  = extract_canonical_blocks(text)
    project_id = text_path.name.split("_")[0]

    document_summary.append({
        "projectid":       project_id,
        "filename":        text_path.name,
        "character_count": len(text),
        "extracted_units": len(blocks_df),
    })

    if blocks_df.empty:
        continue

    blocks_df["projectid"]   = project_id
    blocks_df["source_file"] = text_path.name
    all_extracted_blocks.append(blocks_df)

extracted_df        = pd.concat(all_extracted_blocks, ignore_index=True)
document_summary_df = pd.DataFrame(document_summary)

print("Documents processed:", len(document_summary_df))
print("Extracted units    :", len(extracted_df))
print()
print(extracted_df["type"].value_counts().to_string())

In [ ]:
# --------------------------------------------------
# COVERAGE FLAGS AND SAVE
# --------------------------------------------------

document_summary_df["extraction_status"] = (
    document_summary_df["extracted_units"]
    .apply(lambda x: "success" if x > 0 else "no_units_detected")
)

successful = (document_summary_df["extraction_status"] == "success").sum()
coverage   = successful / len(document_summary_df)

print(f"Successful documents: {successful}/{len(document_summary_df)}")
print(f"Extraction coverage : {coverage:.1%}")
print()

failed_df = document_summary_df[
    document_summary_df["extraction_status"] == "no_units_detected"
].copy()

print("Failed documents:", len(failed_df))

extracted_df.to_csv(PROCESSED_DIR / "pad_components_subcomponents.csv", index=False)
document_summary_df.to_csv(PROCESSED_DIR / "pad_extraction_summary.csv", index=False)
failed_df.to_csv(PROCESSED_DIR / "pad_extraction_failures.csv", index=False)

print("Saved:")
print("  pad_components_subcomponents.csv")
print("  pad_extraction_summary.csv")
print("  pad_extraction_failures.csv")